In essence, we cannot “solve” instances of LWE. But estimating how hard they are is slightly more studied. I think going for a solver is way-way beyond my scope, thus https://github.com/malb/lattice-estimator by Albrecht et al will come in handy.

In [1]:
from sage.crypto.lwe import samples

In [2]:
S = samples(4, 4, 'Regev')
S

[((5, 3, 3, 6), 1),
 ((11, 5, 5, 1), 14),
 ((6, 13, 1, 12), 5),
 ((0, 4, 11, 11), 8)]

In [3]:
# importing submodules with hyphens: https://stackoverflow.com/questions/8350853/
import sys
sys.path.append('lattice-estimator')
import estimator

In [4]:
estimator.LWE.Estimate.rough

<function Estimate.rough at 0x7f704b411bc0>

## Lattice estimator usage with built-in parameter sets

In [5]:
%%time

estimator.schemes.Kyber512

CPU times: user 5 μs, sys: 0 ns, total: 5 μs
Wall time: 6.91 μs


LWEParameters(n=512, q=3329, Xs=D(σ=1.22), Xe=D(σ=1.22), m=512, tag='Kyber 512')

In [6]:
%%time
estimator.LWE.primal_usvp(estimator.schemes.Kyber512)

CPU times: user 78.1 ms, sys: 1.69 ms, total: 79.8 ms
Wall time: 79.4 ms


rop: ≈2^143.8, red: ≈2^143.8, δ: 1.003941, β: 406, d: 998, tag: usvp

In [7]:
%%time
estimator.LWE.estimate.rough(estimator.schemes.Kyber512)

usvp                 :: rop: ≈2^118.6, red: ≈2^118.6, δ: 1.003941, β: 406, d: 998, tag: usvp
dual_hybrid          :: rop: ≈2^115.5, red: ≈2^115.3, guess: ≈2^112.3, β: 395, p: 5, ζ: 0, t: 40, β': 395, N: ≈2^81.4, m: 512
CPU times: user 3.61 s, sys: 3.46 ms, total: 3.62 s
Wall time: 3.64 s


{'usvp': rop: ≈2^118.6, red: ≈2^118.6, δ: 1.003941, β: 406, d: 998, tag: usvp,
 'dual_hybrid': rop: ≈2^115.5, red: ≈2^115.3, guess: ≈2^112.3, β: 395, p: 5, ζ: 0, t: 40, β': 395, N: ≈2^81.4, m: 512}

In [8]:
%%time
#estimator.LWE.estimate(estimator.schemes.Kyber512)

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 4.05 μs


## Lattice estimator with our own parameters
Let's use Regev's methodology for choosing parameters; let's use base parameters similar to Kyber512

In [9]:
S = samples(m=512, n=512, lwe='Regev', seed=4815)

In [10]:
next_prime(pow(512, 2))

262147

In [11]:
from sage.stats.distributions.discrete_gaussian_integer import DiscreteGaussianDistributionIntegerSampler
# this comes very directly from sage.crypto.lwe.Regev


###### NOTE -- I don't think I have seen any paper use this “Both Gaussian” form. Trying just for funsies!
#choose n - nr of variables
n = 512
q = ZZ(next_prime(n**2))
sigma = RR(1/(RR(n).sqrt() * log(n, 2)**2) * q)
arith_mean = 0
# Chi-s - secret distribution
Xs = estimator.ND.DiscreteGaussian(sigma, arith_mean)
# Chi-e - error distribution
Xe = estimator.ND.DiscreteGaussian(sigma, arith_mean)
tag = 'Regev with Gaussian Xs and Gaussian Xe'

#optionally define how many samples we want to limit the oracle to
#sage.all.oo = infinity
m = oo

In [12]:
#attempt 0:
# estimator.lwe_parameters.LWEParameters(n, q, Xs, Xe, m, tag)
#got back:
# AttributeError: '...Sampler' object has no attribute 'resize'
#meaning we probably need to wrap sampler in a latticeEstimator compatible object (from package estimator.nd)

#attempt 1:
Regev_GaussXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_GaussXs_GaussXe

LWEParameters(n=512, q=262147, Xs=D(σ=143.03), Xe=D(σ=143.03), m=+Infinity, tag='Regev with Gaussian Xs and Gaussian Xe')

In [13]:
# the most standard form of Regev: uniform secret, Gaussian error
n = 512
q = ZZ(next_prime(n**2))
sigma = RR(1/(RR(n).sqrt() * log(n, 2)**2) * q)
arith_mean = 0
Xs = estimator.ND.UniformMod(q)
Xe = estimator.ND.DiscreteGaussian(sigma, arith_mean)
tag = 'Regev with uniform Xs and Gaussian Xe'
m = oo

Regev_UniXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_UniXs_GaussXe

LWEParameters(n=512, q=262147, Xs=D(σ=75675.32), Xe=D(σ=143.03), m=+Infinity, tag='Regev with uniform Xs and Gaussian Xe')

In [14]:
n = 512
q = ZZ(next_prime(n**2))
# Chi-s - secret distribution
Xs = estimator.ND.UniformMod(q)
# Chi-e - error distribution
##### manually chose modulus 496 as this would give parameter \sigma close to those with Gaussian distribution
Xe = estimator.ND.UniformMod(ZZ(495))
tag = 'Regev with uniform Xs and uniform Xe'
m = oo

Regev_UniXs_UniXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
Regev_UniXs_UniXe

LWEParameters(n=512, q=262147, Xs=D(σ=75675.32), Xe=D(σ=142.89), m=+Infinity, tag='Regev with uniform Xs and uniform Xe')

### Estimations on our own samples!

In [15]:
deny_list=["arora-gb", "bkw", "bdd", "bdd_hybrid", "bdd_mitm_hybrid", "dual"]

In [16]:
%%time
estimator.LWE.estimate(Regev_GaussXs_GaussXe, deny_list=deny_list)

usvp                 :: rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp
dual_hybrid          :: rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512
CPU times: user 684 ms, sys: 3.01 ms, total: 687 ms
Wall time: 686 ms


{'usvp': rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp,
 'dual_hybrid': rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512}

In [17]:
%%time
estimator.LWE.estimate(Regev_UniXs_GaussXe, deny_list=deny_list)

usvp                 :: rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp
dual_hybrid          :: rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512
CPU times: user 649 ms, sys: 997 μs, total: 650 ms
Wall time: 651 ms


{'usvp': rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1571, tag: usvp,
 'dual_hybrid': rop: ≈2^342.8, red: ≈2^342.2, guess: ≈2^341.3, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.8, m: 512}

In [18]:
%%time
estimator.LWE.estimate(Regev_UniXs_UniXe, deny_list=deny_list)

usvp                 :: rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1562, tag: usvp
dual_hybrid          :: rop: ≈2^342.7, red: ≈2^342.2, guess: ≈2^340.9, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.4, m: 512
CPU times: user 620 ms, sys: 3.89 ms, total: 624 ms
Wall time: 625 ms


{'usvp': rop: ≈2^243.2, red: ≈2^243.2, δ: 1.002508, β: 761, d: 1562, tag: usvp,
 'dual_hybrid': rop: ≈2^342.7, red: ≈2^342.2, guess: ≈2^340.9, β: 1148, p: 3, ζ: 10, t: 0, β': 1070, N: ≈2^221.4, m: 512}

It looks like sometimes the estimator can run for a lot-lot longer than expected. `UniXs_GaussXe` finished in 9.74 seconds wall time, while `UniXs_UniXe` did not estimate in a few minutes... Restarted kernel, then waited 24 minutes, did not finish. Swapped to `estimate.rough()` (from just `estimate()`).

---

Looking back, it was clear what the error was -- the choice of uniform distribution means that going for uniform errors in the entirety of $q$ would be ridiculously difficult. Thus, errors should probably be centered in some smaller subsection. I went back and modified the selection such that my errors would be from an odd modulus which would result a similar $\sigma$ value for the error distribution (in our case, original Gaussian $\chi_e$ had $\sigma\approx 143.03$; uniform $\chi_e$ with modulus $495$ gave $\sigma\approx 142.89$).

## Side-question

It seems that for the noise distribution, stddev is roughly one of the few parameters that matters. Coincidence? Maybe is it the case that for dimensions they care about, these distributions do not differ much?

Maybe the approach for this is to find all the different LWE schemes that lattice-estimator supports? -- Nope, that would be way too many schemes. Rather, maybe try to create a table of what each thing supports.

Information gathered from the lattice-estimator `schemes.py` file -- just so that collecting would be easier. Only focusing on schemes which seemingly have a real implementation behind them.

For Regev and LP-style parameters, used `sage.crypto.lwe.<scheme>` with input values for $n$.

| Scheme           | n     | q       | m (# max samples) | Xs                    | Xe                       | Xe variance | Lowest ROP 2pow   |
|------------------|-------|---------|-------------------|-----------------------|--------------------------|-------------|-------------------|
| Kyber512         | 2*256 | 3329    | 2*256             | CenteredBinom(3)      | CenteredBinom(3)         | 0.00036648  | dual_hybrid 139.7 |
| Kyber768         | 3*256 | 3329    | 3*256             | CenteredBinom(2)      | CenteredBinom(2)         | 0.00030039  | dual_hybrid 196.4 |
| Kyber1024        | 4*256 | 3329    | 4*256             | CenteredBinom(3)      | CenteredBinom(3)         | 0.00036648  | dual_hybrid 262.3 |
| LightSaber       | 2*256 | 8192    | 2*256             | CenteredBinom(5)      | UniformMod(8)            | 0.00027954  | bdd 139.9         |
| Saber            | 3*256 | 8192    | 3*256             | CenteredBinom(4)      | UniformMod(8)            | 0.00027954  | dual_hybrid 204.9 |
| FireSaber        | 4*256 | 8192    | 4*256             | CenteredBinom(3)      | UniformMod(8)            | 0.00027954  | dual_hybrid 267.3 |
| Frodo640         | 640   | 32768   | 640 + 16          | DiscreteGaussian(2.8) | DiscreteGaussian(2.8)    | 0.00008545  | Bdd 162.9         |
| Frodo976         | 976   | 65536   | 976 + 16          | DiscreteGaussian(2.3) | DiscreteGaussian(2.3)    | 0.0000351   | Bdd 225.2         |
| Frodo1344        | 1344  | 65536   | 1344 + 16         | DiscreteGaussian(3.0) | DiscreteGaussian(3.0)    | 0.00004578  | dual_hybrid 281.7 |
| Regev-esque-512  | 2*256 | 262147  | -                 | Uniform               | DiscreteGaussian(57.06)  | 0.00021766  | Bdd 238.8         |
| Regev-esque-768  | 3*256 | 589829  | -                 | Uniform               | DiscreteGaussian(92.42)  | 0.00015669  | Bdd 358.5         |
| Regev-esque-1024 | 4*256 | 1048583 | -                 | Uniform               | DiscreteGaussian(130.72) | 0.00012466  | Bdd 481.1         |
| LP-esque-512     | 2*256 | 4099    | 1152              | DiscreteGaussian(…)   | DiscreteGaussian(2.9)    | 0.00070749  | Bdd 169.2         |
| LP-esque-768     | 3*256 | 8209    | 1664              | DiscreteGaussian(…)   | DiscreteGaussian(3.76)   | 0.00045803  | Bdd 248.2         |
| LP-esque-1024    | 4*256 | 8209    | 2176              | DiscreteGaussian(…)   | DiscreteGaussian(3.53)   | 0.00043002  | Bdd 329.8         |

Trivially, the Regev-style parameters seem way out of scope. 

I will wholeheartedly admit that at this point, I've arrived at a bit of a wall again. I consulted the Copilot sands, which did mention that it might be worth looking at error variance $\alpha_e = \frac{\sigma_e}{q}$ in the table as well. Additionally:
> Kyber and Saber families are operating with astonishingly small noise compared to classical Regev constructions. This is possible because they exploit module structure and have much tighter engineering constraints.

Sands also propose testing a hypothesis:
> Holding (n,q,\alpha) approximately fixed, replacing Gaussian, centered binomial, and bounded uniform error distributions produces only minor changes in estimated attack cost.

And finally:
> ... modern hardness estimates are largely driven by variance (or $\alpha$), whereas distribution shape tends to appear only as a secondary effect except for small-support attacks and proof-related considerations.

In [19]:
# follows sage.crypto.lwe.Regev, wrapping everything in Lattice-Estimator methods
def regev_returner(n):
    q = ZZ(next_prime(n**2))
    sigma = RR(1/(RR(n).sqrt() * log(n, 2)**2) * q)
    arith_mean = 0
    Xs = estimator.ND.UniformMod(q)
    Xe = estimator.ND.DiscreteGaussian(sigma, arith_mean)
    tag = 'Regev with uniform Xs and Gaussian Xe'
    m = oo
    
    Regev_UniXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
    return Regev_UniXs_GaussXe

In [20]:
# follows sage.crypto.lwe.LindnerPeikert, wrapping everything in Lattice-Estimator methods
def lp_returner(n, delta=0.01, m=None):
    if m is None:
        m = 2*n + 128
    # Find c>=1 such that c*exp((1-c**2)/2))**(2*n) == 2**-40
    #         (c*exp((1-c**2)/2))**(2*n) == 2**-40
    #    log((c*exp((1-c**2)/2))**(2*n)) == -40*log(2)
    #       (2*n)*log(c*exp((1-c**2)/2)) == -40*log(2)
    #  2*n*(log(c)+log(exp((1-c**2)/2))) == -40*log(2)
    #            2*n*(log(c)+(1-c**2)/2) == -40*log(2)
    #              2*n*log(c)+n*(1-c**2) == -40*log(2)
    #  2*n*log(c)+n*(1-c**2) + 40*log(2) == 0
    c = SR.var('c')
    c = find_root(2*n*log(c)+n*(1-c**2) + 40*log(2) == 0, 1, 10)
    # Upper bound on s**2/t
    s_t_bound = (sqrt(2) * pi / c / sqrt(2*n*log(2/delta))).n()
    # Interpretation of "choose q just large enough to allow for a Gaussian parameter s>=8" in [LP2011]_
    q = next_prime(floor(2**round(log(256 / s_t_bound, 2))))
    # Gaussian parameter as defined in [LP2011]_
    s = sqrt(s_t_bound*floor(q/4))
    # Transform s into stddev
    stddev = s/sqrt(2*pi.n())
    
    Xe = estimator.ND.DiscreteGaussian(stddev)
    ## TODO: might have to edit Xs
    Xs = estimator.ND.DiscreteGaussian(stddev)
    tag = 'LP with DiscreteGaussian(stddev) Xs and DiscreteGaussian(stddev) Xe (probably wrong)'

    LP_GaussXs_GaussXe = estimator.lwe_parameters.LWEParameters(n=n, q=q, Xs=Xs, Xe=Xe, m=m, tag=tag)
    return LP_GaussXs_GaussXe

In [21]:
regev_returner(512)

LWEParameters(n=512, q=262147, Xs=D(σ=75675.32), Xe=D(σ=143.03), m=+Infinity, tag='Regev with uniform Xs and Gaussian Xe')

In [22]:
print('=======================')
print('======BEGIN KYBER======')
print('---512---')
estimator.LWE.estimate(estimator.schemes.Kyber512)
print('---768---')
estimator.LWE.estimate(estimator.schemes.Kyber768)
print('---1024---')
estimator.LWE.estimate(estimator.schemes.Kyber1024)
print('=======================')
print('======BEGIN SABER======')
print('---512---')
estimator.LWE.estimate(estimator.schemes.LightSaber)
print('---768---')
estimator.LWE.estimate(estimator.schemes.Saber)
print('---1024---')
estimator.LWE.estimate(estimator.schemes.FireSaber)
print('=======================')
print('======BEGIN FRODO======')
print('---640---')
estimator.LWE.estimate(estimator.schemes.Frodo640)
print('---976---')
estimator.LWE.estimate(estimator.schemes.Frodo976)
print('---1344---')
estimator.LWE.estimate(estimator.schemes.Frodo1344)
print('=============================')
print('======BEGIN REGEV-ESQUE======')
print('---512---')
estimator.LWE.estimate(regev_returner(512))
print('---768---')
estimator.LWE.estimate(regev_returner(768))
print('---1024---')
estimator.LWE.estimate(regev_returner(1024))
print('==========================')
print('======BEGIN LP-ESQUE======')
print('---512---')
estimator.LWE.estimate(lp_returner(512))
print('---768---')
estimator.LWE.estimate(lp_returner(768))
print('---1024---')
estimator.LWE.estimate(lp_returner(1024))


======BEGIN KYBER======
---512---
bkw                  :: rop: ≈2^178.8, m: ≈2^166.8, mem: ≈2^167.8, b: 14, t1: 0, t2: 16, ℓ: 13, #cod: 448, #top: 0, #test: 64, tag: coded-bkw
usvp                 :: rop: ≈2^143.8, red: ≈2^143.8, δ: 1.003941, β: 406, d: 998, tag: usvp
bdd                  :: rop: ≈2^140.2, red: ≈2^139.1, svp: ≈2^139.3, β: 389, η: 422, d: 1005, tag: bdd
dual                 :: rop: ≈2^149.9, mem: ≈2^97.1, m: 512, β: 424, d: 1024, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^139.7, red: ≈2^139.5, guess: ≈2^135.9, β: 387, p: 5, ζ: 0, t: 50, β': 391, N: ≈2^81.1, m: 512
---768---
bkw                  :: rop: ≈2^238.3, m: ≈2^225.5, mem: ≈2^226.5, b: 19, t1: 1, t2: 17, ℓ: 18, #cod: 653, #top: 1, #test: 95, tag: coded-bkw
usvp                 :: rop: ≈2^204.9, red: ≈2^204.9, δ: 1.002902, β: 624, d: 1427, tag: usvp
bdd                  :: rop: ≈2^201.0, red: ≈2^199.9, svp: ≈2^200.0, β: 606, η: 640, d: 1420, tag: bdd
dual                 :: rop: ≈2^214.3, mem: ≈2^142.8, m: 72

{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'bkw': rop: ≈2^432.2, m: ≈2^419.2, mem: ≈2^420.2, b: 32, t1: 0, t2: 17, ℓ: 31, #cod: 923, #top: 0, #test: 104, tag: coded-bkw,
 'usvp': rop: ≈2^334.6, red: ≈2^334.6, δ: 1.001919, β: 1086, d: 2141, tag: usvp,
 'bdd': rop: ≈2^329.8, red: ≈2^328.8, svp: ≈2^328.8, β: 1065, η: 1100, d: 2155, tag: bdd,
 'bdd_hybrid': rop: ≈2^329.8, red: ≈2^328.8, svp: ≈2^328.8, β: 1065, η: 1100, ζ: 0, |S|: 1, d: 2178, prob: 1.0, ↻: 1, tag: hybrid,
 'bdd_mitm_hybrid': rop: ≈2^875.2, red: ≈2^875.2, svp: ≈2^562.4, β: 1087, η: 2, ζ: 0, |S|: 1, d: 2195, prob: ≈2^-538.0, ↻: ≈2^540.2, tag: hybrid,
 'dual': rop: ≈2^351.9, mem: ≈2^240.0, m: 1215, β: 1144, d: 2239, ↻: 1, tag: dual,
 'dual_hybrid': rop: ≈2^342.7, red: ≈2^342.7, guess: ≈2^319.2, β: 1112, p: 3, ζ: 20, t: 0, β': 1077, N: ≈2^222.3, m: 1024}

---

from LWE BKW:

    The returned cost dictionary has the following entries:
    - ``rop``: Total number of word operations (≈ CPU cycles).
    - ``b``: BKW tables have size `q^b`.
    - ``t1``: Number of plain BKW tables.
    - ``t2``: Number of Coded-BKW tables.
    - ``ℓ``: Hypothesis testing has tables of size `q^{ℓ+1}`
    - ``#cod``: Number of coding steps.
    - ``#top``: Number of guessing steps (typically zero)
    - ``#test``: Number of coordinates to do hypothesis testing on.

from LWE dual:

    - ``rop``: Total number of word operations (≈ CPU cycles).
    - ``mem``: Total amount of memory used by solver (in elements mod q).
    - ``red``: Number of word operations in lattice reduction.
    - ``δ``: Root-Hermite factor targeted by lattice reduction.
    - ``β``: BKZ block size.
    - ``ζ``: Number of guessed coordinates.
    - ``h1``: Number of non-zero components among guessed coordinates (if secret distribution is sparse)
    - ``prob``: Probability of success in guessing.
    - ``repetitions``: How often we are required to repeat the attack.
    - ``d``: Lattice dimension.
    - ``t``: Number of secrets to guess mod 2 (only if ``fft`` is ``True``)
    
from LWE guess:

    - ``rop``: Total number of word operations (≈ CPU cycles).
    - ``mem``: memory requirement in integers mod q.
    - ``m``: Required number of samples to distinguish the correct solution with high probability.
    - ``k``: Splitting dimension.
    - ``↻``: Repetitions required to achieve targeted success probability

from LWE primal: 

    - ``rop``: Total number of word operations (≈ CPU cycles).
    - ``red``: Number of word operations in lattice reduction.
    - ``δ``: Root-Hermite factor targeted by lattice reduction.
    - ``β``: BKZ block size.
    - ``η``: Dimension of the final BDD call.
    - ``ζ``: Number of guessed coordinates.
    - ``|S|``: Guessing search space.
    - ``prob``: Probability of success in guessing.
    - ``repeat``: How often to repeat the attack.
    - ``d``: Lattice dimension.
